# Step 3 — Paper-grade numbers: corrected tests + literature validation

**Click Runtime ▸ Run all.** Takes about 15–25 minutes. In **Cell 6** a *Choose files* button appears:
- upload your completed `literature_validation.csv`, **or**
- click *Cancel upload* to use the 5 built-in DRAFT rows (this only demonstrates the code; 5 rows is far too few for a paper).

What this does:
1. Re-runs the chemical-system test with the **real hard rule** (a compound is called a metal if P(non-metal) ≤ 0.5), with 95% confidence intervals that resample whole chemical systems.
2. Reports performance **per anion family**.
3. Redoes the blind test on known solar materials with the **corrected** rule.
4. Checks the model against **measured gaps from the literature** that are not in the training data.

If any cell turns red, **stop and send me the last 5 lines.**

## Cell 1 — Install
If Colab shows *Restart session*, click it, then Run all again.

In [ ]:
%pip install -q matminer pymatgen scikit-learn pandas numpy scipy matplotlib

## Cell 2 — Load experimental data, build features

In [ ]:
import os, re, json, shutil, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

SEED = 42
OUT = "step3_outputs"
os.makedirs(OUT, exist_ok=True)

from matminer.datasets import load_dataset
raw = load_dataset("matbench_expt_gap")
target_col = [c for c in raw.columns if "gap" in str(c).lower()][0]
formula_col = [c for c in raw.columns if c != target_col][0]
df = pd.DataFrame({"formula": raw[formula_col].astype(str), "gap": raw[target_col].astype(float)})
print("Rows:", len(df), "(expected 4604)")
assert len(df) > 4000, "Dataset looks wrong - stop and send me this message."

TOKEN = re.compile(r"([A-Z][a-z]?)(\d*\.?\d*)")
def parse_formula(f):
    comp = {}
    for el, n in TOKEN.findall(f):
        comp[el] = comp.get(el, 0.0) + (float(n) if n else 1.0)
    return comp
def chem_system(f): return "-".join(sorted(parse_formula(f)))
def frac_key(f):
    c = parse_formula(f); t = sum(c.values())
    return tuple(sorted((e, round(n / t, 5)) for e, n in c.items()))
CHALC, HAL, PNICT = {"S","Se","Te"}, {"F","Cl","Br","I"}, {"N","P","As","Sb"}
def family(f):
    els = set(parse_formula(f))
    if "O" in els: return "oxide"
    if els & CHALC: return "chalcogenide"
    if els & HAL: return "halide"
    if els & PNICT: return "pnictide"
    return "other"

from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty
try:
    EP = ElementProperty.from_preset("magpie", impute_nan=True)
except TypeError:
    EP = ElementProperty.from_preset("magpie")
def featurize(formulas):
    rows = [EP.featurize(Composition(parse_formula(f))) for f in formulas]
    return np.nan_to_num(np.array(rows, dtype=float))

df = df[df.formula.apply(lambda f: len(parse_formula(f)) > 0)].reset_index(drop=True)
df["system"] = df.formula.apply(chem_system)
df["family"] = df.formula.apply(family)
df["key"] = df.formula.apply(frac_key)
y = df.gap.values
nm = (y > 0).astype(int)
print("Computing chemical features (1-2 minutes)...")
X = featurize(df.formula.tolist())
TRAIN_KEYS = set(df.key)
print("Features:", X.shape, "| non-metal share: %.2f" % nm.mean())

## Cell 3 — Chemical-system test with confidence intervals and per-family results

In [ ]:
N_FOLDS = 5
def grouped_folds(groups, seed=SEED):
    rng = np.random.default_rng(seed)
    cnt = Counter(groups); keys = list(cnt); rng.shuffle(keys); keys.sort(key=lambda k: -cnt[k])
    load = np.zeros(N_FOLDS); assign = {}
    for k in keys:
        j = int(np.argmin(load)); assign[k] = j; load[j] += cnt[k]
    fo = np.array([assign[g] for g in groups])
    return [(np.where(fo != j)[0], np.where(fo == j)[0]) for j in range(N_FOLDS)]
def rf_reg(n=200): return RandomForestRegressor(n_estimators=n, n_jobs=-1, random_state=SEED)
def rf_clf(n=200): return RandomForestClassifier(n_estimators=n, n_jobs=-1, random_state=SEED)
def p_nonmetal(clf, Xq):
    return clf.predict_proba(Xq)[:, list(clf.classes_).index(1)] if 1 in clf.classes_ else np.zeros(len(Xq))

def cluster_mean_ci(vals, clusters, n_boot=2000, seed=SEED):
    """Mean with 95% CI, resampling whole chemical systems (rows inside a system are not independent)."""
    vals = np.asarray(vals, float)
    u, inv = np.unique(clusters, return_inverse=True)
    s = np.bincount(inv, weights=vals); c = np.bincount(inv)
    rng = np.random.default_rng(seed); k = len(u); b = np.empty(n_boot)
    for i in range(n_boot):
        pick = rng.integers(0, k, k); b[i] = s[pick].sum() / c[pick].sum()
    return float(vals.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))

groups = df.system.values
single = np.zeros(len(df)); regp = np.zeros(len(df)); pnm = np.zeros(len(df))
for k, (tr, te) in enumerate(grouped_folds(groups)):
    assert not (set(groups[tr]) & set(groups[te])), "leak"
    single[te] = rf_reg().fit(X[tr], y[tr]).predict(X[te])
    trn = tr[nm[tr] == 1]
    regp[te] = rf_reg().fit(X[trn], y[trn]).predict(X[te])
    pnm[te] = p_nonmetal(rf_clf().fit(X[tr], nm[tr]), X[te])
    print("fold", k + 1, "of", N_FOLDS, "done", flush=True)

hard = np.where(pnm > 0.5, regp, 0.0)               # the real two-stage rule
e_single, e_hard, e_reg = np.abs(y - single), np.abs(y - hard), np.abs(y - regp)
m1 = nm == 1
rows = []
def add(name, v, cl): m, lo, hi = cluster_mean_ci(v, cl); rows.append(dict(metric=name, value=m, ci_low=lo, ci_high=hi))
add("MAE all compounds - single-stage (eV)", e_single, groups)
add("MAE all compounds - two-stage hard rule (eV)", e_hard, groups)
add("MAE difference single minus two-stage (eV)", e_single - e_hard, groups)
add("MAE true non-metals - single-stage (eV)", e_single[m1], groups[m1])
add("MAE true non-metals - two-stage hard rule (eV)", e_hard[m1], groups[m1])
add("MAE true non-metals - regressor if non-metal known (eV)", e_reg[m1], groups[m1])
add("Metal vs non-metal accuracy", ((pnm > 0.5).astype(int) == nm).astype(float), groups)
MAIN = pd.DataFrame(rows).round(3); MAIN.to_csv(f"{OUT}/main_results.csv", index=False)
print(MAIN.to_string(index=False))
print("Metal vs non-metal AUC: %.3f" % roc_auc_score(nm, pnm))

fam_rows = []
for g in sorted(df.family.unique()):
    mk = df.family.values == g; mn = mk & m1
    r = dict(family=g, n=int(mk.sum()), nonmetal_share=round(float(nm[mk].mean()), 2),
             accuracy=round(float(((pnm[mk] > 0.5).astype(int) == nm[mk]).mean()), 3))
    if mn.sum() >= 20:
        m, lo, hi = cluster_mean_ci(e_reg[mn], groups[mn]); r.update(regressor_MAE=round(m, 3), ci_low=round(lo, 3), ci_high=round(hi, 3))
        r["two_stage_MAE_nonmetals"] = round(float(e_hard[mn].mean()), 3)
    fam_rows.append(r)
FAM = pd.DataFrame(fam_rows); FAM.to_csv(f"{OUT}/per_family_system_protocol.csv", index=False)
print("\nPer-family results (chemical-system folds):\n", FAM.to_string(index=False))

RES = (y - regp)[m1]                                  # out-of-fold residuals on true non-metals
Q90 = float(np.quantile(np.abs(RES), 0.9))
print(f"\n90% of non-metal gap errors are below {Q90:.2f} eV")

## Cell 4 — Final models and the CORRECTED blind test on known solar materials
For each compound the model is retrained without any compound sharing its chemical system. A compound with P(non-metal) ≤ 0.5 is called a **metal (gap = 0)** — the real rule.

In [ ]:
REG = rf_reg(300).fit(X[m1], y[m1]); CLF = rf_clf(300).fit(X, nm)

def window_prob(pred, pn):
    t = pred[:, None] + RES[None, :]
    return pn * ((t >= 1.0) & (t <= 1.8)).mean(axis=1)

def predict_with(reg, clf, formulas):
    xq = featurize(formulas); pn = p_nonmetal(clf, xq); pr = reg.predict(xq)
    return pn, pr, np.where(pn > 0.5, pr, 0.0), window_prob(pr, pn)

KNOWN = {"Cu2ZnSnS4": 1.5, "Zn3P2": 1.5, "FeS2": 0.95, "SnS": 1.3, "Cu2O": 2.1, "BaZrS3": 1.8, "Fe2O3": 2.1}
rows = []
for f, lit in KNOWN.items():
    keep = df.system.values != chem_system(f)
    reg = rf_reg().fit(X[keep & m1], y[keep & m1]); clf = rf_clf().fit(X[keep], nm[keep])
    sng = float(rf_reg().fit(X[keep], y[keep]).predict(featurize([f]))[0])
    pn, pr, hd, _ = predict_with(reg, clf, [f])
    rows.append(dict(compound=f, literature_gap_approx=lit, p_nonmetal=round(float(pn[0]), 2),
                     regressor_if_nonmetal=round(float(pr[0]), 2), TWO_STAGE_HARD=round(float(hd[0]), 2),
                     single_stage=round(sng, 2), err_two_stage_hard=round(abs(hd[0] - lit), 2),
                     err_single=round(abs(sng - lit), 2), rows_removed=int((~keep).sum())))
BL = pd.DataFrame(rows); BL.to_csv(f"{OUT}/blind_recovery_corrected.csv", index=False)
print(BL.to_string(index=False))
print("\nMean error  two-stage (hard rule): %.2f eV | single-stage: %.2f eV" % (BL.err_two_stage_hard.mean(), BL.err_single.mean()))
print("Literature gaps here are approximate orientation values, not a validation set.")

## Cell 5 — Literature file
These rows are **DRAFT**: each value was read from the cited page but must be **verified by you** against the paper (value, measurement, DOI). Your completed file replaces them in Cell 6.

In [ ]:
LIT_DEFAULT = pd.DataFrame([
 dict(formula="SnS", measured_gap_eV=1.37, phase="co-evaporated thin film", method="optical (method not stated on page read)", temperature_K=np.nan,
      doi="10.1007/s00339-005-3475-y", status="DRAFT-verify"),
 dict(formula="Cu2SnS3", measured_gap_eV=1.06, phase="co-evaporated absorber in solar cell", method="estimated from quantum-efficiency measurement", temperature_K=np.nan,
      doi="10.1016/j.jallcom.2016.07.293", status="DRAFT-verify"),
 dict(formula="Cu2FeSnS4", measured_gap_eV=1.30, phase="stannite thin film", method="UV-Vis-NIR absorbance", temperature_K=np.nan,
      doi="10.1007/s10854-017-6469-6", status="DRAFT-verify"),
 dict(formula="Cu2FeSnS4", measured_gap_eV=1.34, phase="wurtzite thin film", method="UV-Vis-NIR absorbance", temperature_K=np.nan,
      doi="10.1007/s10854-017-6469-6", status="DRAFT-verify"),
 dict(formula="FePS3", measured_gap_eV=1.40, phase="bulk crystal", method="optical absorption (Tauc) + photoacoustic", temperature_K=300,
      doi="arXiv:2108.01742 (journal DOI to verify)", status="DRAFT-verify"),
])
print(LIT_DEFAULT.to_string(index=False))

## Cell 6 — Validate against measured literature gaps
A *Choose files* button appears. Upload your completed `literature_validation.csv`, or click *Cancel upload* to use the draft rows.

In [ ]:
lit = LIT_DEFAULT.copy(); source_used = "built-in DRAFT rows"
try:
    from google.colab import files
    up = files.upload()
    if up:
        name = list(up)[0]; lit = pd.read_csv(name); source_used = name
except ImportError:
    if os.path.exists("literature_validation.csv"):
        lit = pd.read_csv("literature_validation.csv"); source_used = "literature_validation.csv"
print("Literature source:", source_used)

lit["measured_gap_eV"] = pd.to_numeric(lit["measured_gap_eV"], errors="coerce")
lit = lit.dropna(subset=["formula", "measured_gap_eV"])
lit = lit[lit.formula.astype(str).apply(lambda f: len(parse_formula(f)) > 0)].copy()
lit["key"] = lit.formula.astype(str).apply(frac_key)
in_train = lit.key.isin(TRAIN_KEYS)
print("Rows with a measured value:", len(lit), "| already in the training data (excluded):", int(in_train.sum()))
lit = lit[~in_train]
agg = lit.groupby("formula").agg(measured_mean=("measured_gap_eV", "mean"), measured_min=("measured_gap_eV", "min"),
                                 measured_max=("measured_gap_eV", "max"), n_values=("measured_gap_eV", "size")).reset_index()
N = len(agg)
print("Distinct materials used for validation:", N)
if N == 0:
    print("No usable rows - fill the CSV and run this cell again.")
else:
    pn, pr, hd, pw = predict_with(REG, CLF, agg.formula.tolist())
    agg["p_nonmetal"] = pn.round(2); agg["pred_gap_eV"] = hd.round(2); agg["regressor_gap_eV"] = pr.round(2)
    agg["p_in_window"] = pw.round(2)
    agg["abs_error_eV"] = (agg.pred_gap_eV - agg.measured_mean).abs().round(2)
    agg["measured_within_90pct_band"] = ((agg.measured_mean >= (pr - Q90)) & (agg.measured_mean <= (pr + Q90)))
    agg["family"] = agg.formula.apply(family)
    agg.to_csv(f"{OUT}/literature_validation_results.csv", index=False)
    print(agg[["formula","family","measured_mean","pred_gap_eV","p_nonmetal","p_in_window","abs_error_eV","measured_within_90pct_band"]].to_string(index=False))

    err = (agg.pred_gap_eV - agg.measured_mean).values
    lv = dict(n_materials=int(N), MAE_eV=float(np.abs(err).mean()), bias_pred_minus_measured_eV=float(err.mean()),
              within_0p5_eV=float((np.abs(err) <= 0.5).mean()), measured_inside_90pct_band=float(agg.measured_within_90pct_band.mean()))
    truth = agg.measured_mean.between(1.0, 1.8).values; rec = (agg.p_in_window >= 0.30).values
    lv.update(window_TP=int((truth & rec).sum()), window_FP=int((~truth & rec).sum()),
              window_FN=int((truth & ~rec).sum()), window_TN=int((~truth & ~rec).sum()))
    if N >= 8:
        rng = np.random.default_rng(SEED); bs = [np.abs(err[rng.integers(0, N, N)]).mean() for _ in range(2000)]
        lv["MAE_ci95"] = [float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))]
    print("\n", json.dumps(lv, indent=1))
    if N < 15:
        print("\nWARNING: only %d materials. This is NOT enough for a paper-grade validation (aim for 30-60)." % N)
    fig, ax = plt.subplots(figsize=(4.4, 4.2))
    ax.scatter(agg.measured_mean, agg.pred_gap_eV, s=30)
    ax.plot([0, 4], [0, 4], "k--", lw=.8); ax.axvspan(1.0, 1.8, color="orange", alpha=.15)
    ax.set_xlabel("Measured gap, literature (eV)"); ax.set_ylabel("Predicted gap (eV)"); ax.set_xlim(0, 4); ax.set_ylim(0, 4)
    fig.tight_layout(); fig.savefig(f"{OUT}/fig_literature_validation.png", dpi=150); plt.show()
    LV = lv

## Cell 7 — Save and download

In [ ]:
summary = dict(main_results=MAIN.to_dict(orient="records"), per_family=FAM.to_dict(orient="records"),
               error90_nonmetals_eV=Q90, blind_corrected=BL.to_dict(orient="records"),
               blind_mean_err_two_stage_hard=float(BL.err_two_stage_hard.mean()), blind_mean_err_single=float(BL.err_single.mean()),
               literature_source=source_used, literature=(LV if N > 0 else None), auc=float(roc_auc_score(nm, pnm)))
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=2, default=float)
shutil.make_archive("step3_results", "zip", OUT)
print("Saved step3_results.zip")
try:
    from google.colab import files
    files.download("step3_results.zip")
except ImportError:
    pass